# 02_eda.ipynb
**Purpose:** Exploratory Data Analysis — distributions, correlations, and bivariate plots.

| | |
|---|---|
| **Input** | `data/analysis_clean.csv` — output of `01_merge.ipynb` |
| **Output** | `output/fig_wage_dist.png`, `output/fig_log_wage.png`, `output/fig_depression.png`, `output/fig_education.png`, `output/fig_hours.png`, `output/fig_wage_by_educ.png`, `output/fig_depression_vs_wage.png`, `output/fig_correlation.png` |
| **Key functions** | `load_clean()`, `save_fig()`, `plot_heat_map()`, `plot_wage_distribution()`, `plot_log_wage()`, `plot_depression()`, `plot_education()`, `plot_hours()`, `plot_wage_by_education()`, `plot_depression_vs_wage()` | 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
#Paths
ROOT       = Path.cwd().parent
DATA_DIR   = ROOT / "data"
OUTPUT_DIR = ROOT / "output"

INPUT_FILE  = DATA_DIR / "analysis_clean.csv"   # adjust per notebook
OUTPUT_FILE = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDUC_LABELS = {1: '< HS', 2: 'HS diploma', 3: 'Some college',
               4: 'AA degree', 5: 'BA degree', 6: 'Some grad',
               7: 'MA/MS', 8: 'PhD/EdD'}

HRS_LABELS  = {1: '≤20', 2: '21-30', 3: '31-35', 4: '36-39', 5: '40', 6: '>40'}

In [3]:
#functions
def load_clean(path: Path) -> pd.DataFrame:
    """Load the clean analysis CSV."""
    df = pd.read_csv(path)
    print(f"[load_clean] Shape: {df.shape}")
    return df


def save_fig(fig: plt.Figure, filename: str) -> None:
    """Save figure to output dir and close it."""
    dest = OUTPUT_DIR / filename
    fig.savefig(dest, bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"[save_fig] Saved → {dest}")

def plot_heat_map(df :pd.DataFrame) -> None:
    """ Correlation Matrix of Key Variables."""
    corr_cols = ['hourly_wage', 'depression_score', 'hh_income', 'year_born', 'experience']
    corr_df = df[corr_cols].apply(pd.to_numeric, errors='coerce')
    sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix of Key Variables')
    plt.tight_layout()
    save_fig(plt.gcf(), "fig_correlation.png")


def plot_wage_distribution(df: pd.DataFrame) -> None:
    """Histogram of hourly wage with median line."""
    df['hourly_wage'] = pd.to_numeric(df['hourly_wage'], errors='coerce')
    df.dropna(subset=['hourly_wage'], inplace=True)
    plt.figure(figsize=(8, 5))
    sns.histplot(df['hourly_wage'], bins=50, color='steelblue')
    plt.axvline(df['hourly_wage'].median(), color='orange', linestyle='--', label='Median')
    plt.title('Hourly Wage Distribution')
    plt.legend()
    save_fig(plt.gcf(), 'fig_wage_dist.png')


def plot_log_wage(df: pd.DataFrame) -> None:
    """Histogram of log-transformed hourly wage."""
    plt.figure(figsize=(8, 5))
    plt.hist(np.log1p(df['hourly_wage'].dropna()), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    plt.xlabel("Log(Hourly Wage + 1)")
    plt.ylabel("Count")
    plt.title("Log-Transformed")
    save_fig(plt.gcf(), 'fig_log_wage.png')


def plot_depression(df: pd.DataFrame) -> None:
    """Histogram of depression score with clinical threshold."""
    df['depression_score'] = pd.to_numeric(df['depression_score'], errors='coerce')
    plt.figure(figsize=(8,5))
    sns.histplot(df['depression_score'], bins=30, color='mediumpurple')
    plt.axvline(10, color = 'red', linestyle="--", label="Clinical threshold")
    plt.title('Depression Score')
    plt.legend()
    save_fig(plt.gcf(), 'fig_depression.png')


def plot_education(df: pd.DataFrame, educ_labels: dict) -> None:
    """Bar chart of education level counts."""
    counts = df['education'].value_counts().sort_index()
    counts.index = [educ_labels.get(i, i) for i in counts.index]
    plt.figure(figsize=(8,5))
    plt.bar(counts.index, counts.values, color='steelblue')
    plt.title('Education Level')
    plt.tick_params(axis='x', rotation=30)
    save_fig(plt.gcf(), 'fig_education.png')


def plot_hours(df: pd.DataFrame, hrs_labels: dict) -> None:
    """Bar chart of weekly hours worked categories."""
    counts = df['hours_cat'].value_counts().sort_index()
    counts.index = [hrs_labels.get(i, i) for i in counts.index]
    plt.figure(figsize=(8,5))
    plt.bar(counts.index, counts.values, color='green')
    plt.title('Hours Worked Per Week')
    save_fig(plt.gcf(), 'fig_hours.png')


def plot_wage_by_education(df: pd.DataFrame, educ_labels: dict) -> None:
    """Boxplot of hourly wage by education level."""
    educ_order = sorted(df['education'].dropna().unique())
    plt.boxplot(
        [df[df['education'] == e]['hourly_wage'] for e in educ_order],
        labels=[educ_labels.get(e, str(e)) for e in educ_order]
    )
    plt.title('Wage by Education Level')
    plt.tick_params(axis='x', rotation=30)
    save_fig(plt.gcf(), 'fig_wage_by_educ.png')


def plot_depression_vs_wage(df: pd.DataFrame) -> None:
    """Scatter: depression score vs hourly wage."""
    plt.figure(figsize=(8,5))
    plt.scatter(df['depression_score'], df['hourly_wage'],
               alpha=0.1, s=8, color='mediumpurple')
    plt.xlabel('CESD-7 Score')
    plt.ylabel('Hourly Wage')
    plt.title('Depression Score vs. Wage')
    save_fig(plt.gcf(), 'fig_depression_vs_wage.png')

In [4]:
#before plotting
df = load_clean(INPUT_FILE)
print("\n── BEFORE EDA ──")
print(df[['hourly_wage', 'depression_score', 'experience',
          'hh_income', 'interaction_quality_total']].describe().round(2))

[load_clean] Shape: (5192, 22)

── BEFORE EDA ──
                                              hourly_wage depression_score  \
count                                                5192             5192   
unique                                               1277               40   
top     WF spawned from center with only administrativ...              0.0   
freq                                                  483             1724   

       experience           hh_income interaction_quality_total  
count        5192                5192                      5192  
unique          8                   7                         7  
top     <=5 years  $15,001 to $30,000         All items correct  
freq         1279                1209                      2373  


In [5]:
#run all the plots
plot_heat_map(df)
plot_wage_distribution(df)
plot_log_wage(df)
plot_depression(df)
plot_education(df, EDUC_LABELS)
plot_hours(df, HRS_LABELS)
plot_wage_by_education(df, EDUC_LABELS)
plot_depression_vs_wage(df)


print("\n── AFTER EDA ── all figures written to output/")

[save_fig] Saved → C:\Wage Project\output\fig_correlation.png
[save_fig] Saved → C:\Wage Project\output\fig_wage_dist.png
[save_fig] Saved → C:\Wage Project\output\fig_log_wage.png
[save_fig] Saved → C:\Wage Project\output\fig_depression.png
[save_fig] Saved → C:\Wage Project\output\fig_education.png
[save_fig] Saved → C:\Wage Project\output\fig_hours.png


C:\Users\ryan5\AppData\Local\Temp\ipykernel_17948\1105005721.py:83: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(


[save_fig] Saved → C:\Wage Project\output\fig_wage_by_educ.png
[save_fig] Saved → C:\Wage Project\output\fig_depression_vs_wage.png

── AFTER EDA ── all figures written to output/
